In [1]:
import os

In [2]:
%pwd

'/workspaces/MLOps-Car-Price-Prediction-System/notebooks'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/workspaces/MLOps-Car-Price-Prediction-System'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train_data_file_path: str
    target_column: str
    mlflow_tracking_uri: str
    experiment_name: str

In [6]:
from Car_Price_Prediction_System.constants import *
from Car_Price_Prediction_System.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(self, config_filepath=Config_Filepath, params_filepath=Params_Filepath, schema_filepath=Schema_Filepath):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
    
        create_directories([self.config.artifacts_root])
    
    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir = config.root_dir,
            train_data_file_path = config.train_data_file_path,
            target_column=schema.name,
            mlflow_tracking_uri=config.mlflow_tracking_uri,
            experiment_name=config.experiment_name
        )

        return model_trainer_config

In [8]:
from Car_Price_Prediction_System import logger
import pandas as pd
import joblib
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor)
import mlflow
from sklearn.model_selection import GridSearchCV

In [9]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config
        self.params = read_yaml(Params_Filepath)

    def train(self):
        logger.info("Loading Training Dataset")
        train_df = pd.read_csv(self.config.train_data_file_path)

        X_train = train_df.drop(columns=[self.config.target_column])
        y_train = train_df[self.config.target_column]

        models = {
            "RandomForestRegressor": RandomForestRegressor(),
            "GradientBoostingRegressor": GradientBoostingRegressor(),
            "AdaBoostRegressor": AdaBoostRegressor(),
            "ExtraTreesRegressor": ExtraTreesRegressor()
        }

        mlflow.set_tracking_uri(self.config.mlflow_tracking_uri)
        mlflow.set_experiment(self.config.experiment_name)

        for model_name, model in models.items():
            logger.info(f"Training {model_name}")
            grid_search = GridSearchCV(
                estimator=model,
                param_grid=dict(self.params[model_name]),
                scoring="r2",
                cv=5,
                n_jobs=-1
            )
            logger.info(self.params[model_name])

            with mlflow.start_run(run_name=model_name):
                grid_search.fit(X_train, y_train)

                score = grid_search.best_score_
                model = grid_search.best_estimator_

                logger.info(f"{model_name} CV Score : {score}")

                mlflow.log_params(grid_search.best_params_)
                mlflow.log_metric("cv_r2_score", score)

                mlflow.sklearn.log_model(sk_model=model, name="model")

                local_model_path = os.path.join(self.config.root_dir, f"{model_name}.joblib")

                joblib.dump(model, local_model_path)
                logger.info(f"{model_name} Saved successfully")

In [10]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer_config = ModelTrainer(config=model_trainer_config)
    model_trainer_config.train()
except Exception as e:
    raise e

[2026-07-23 05:49:30,118] : INFO : common : Created File: <_io.TextIOWrapper name='config/config.yaml' mode='r' encoding='UTF-8'>
[2026-07-23 05:49:30,126] : INFO : common : Created File: <_io.TextIOWrapper name='config/params.yaml' mode='r' encoding='UTF-8'>
[2026-07-23 05:49:30,130] : INFO : common : Created File: <_io.TextIOWrapper name='config/schema.yaml' mode='r' encoding='UTF-8'>
[2026-07-23 05:49:30,132] : INFO : common : Created Directory: artifacts
[2026-07-23 05:49:30,133] : INFO : common : Created Directory: artifacts/model_trainer
[2026-07-23 05:49:30,140] : INFO : common : Created File: <_io.TextIOWrapper name='config/params.yaml' mode='r' encoding='UTF-8'>
[2026-07-23 05:49:30,141] : INFO : 1156819769 : Loading Training Dataset


2026/07/23 05:49:33 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/23 05:49:33 INFO mlflow.store.db.utils: Updating database tables
2026/07/23 05:49:35 INFO mlflow.tracking.fluent: Experiment with name 'Car_Price_Prediction' does not exist. Creating a new experiment.


[2026-07-23 05:49:35,558] : INFO : 1156819769 : Training RandomForestRegressor
[2026-07-23 05:49:35,561] : INFO : 1156819769 : {'n_estimators': [100, 200], 'criterion': ['squared_error'], 'max_depth': [10, 15, 20], 'min_samples_split': [2, 5], 'min_samples_leaf': [1, 2], 'max_features': ['sqrt', 'log2'], 'n_jobs': [-1]}
[2026-07-23 05:50:29,682] : INFO : 1156819769 : RandomForestRegressor CV Score : 0.8680197181033172
[2026-07-23 05:50:45,991] : INFO : 1156819769 : RandomForestRegressor Saved successfully
[2026-07-23 05:50:46,012] : INFO : 1156819769 : Training GradientBoostingRegressor
[2026-07-23 05:50:46,014] : INFO : 1156819769 : {'n_estimators': [100, 200], 'learning_rate': [0.01, 0.05, 0.1], 'max_depth': [3, 5], 'subsample': [0.8, 1.0]}
[2026-07-23 05:50:56,898] : INFO : 1156819769 : GradientBoostingRegressor CV Score : 0.9097175639775713
[2026-07-23 05:51:06,654] : INFO : 1156819769 : GradientBoostingRegressor Saved successfully
[2026-07-23 05:51:06,663] : INFO : 1156819769 : Tr